# Data catalog usage

In [1]:
# ---- Imports ----
from __future__ import annotations
import os
import urllib3
import polars as pl
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime

from pyiceberg.catalog import load_catalog
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThan
from pyarrow.fs import S3FileSystem
from pyiceberg.io.pyarrow import PyArrowFileIO

import truststore
truststore.inject_into_ssl()

In [2]:
# ---- Utils funcntions 
# parsing bools
def env_bool(name: str, default: str = "false") -> bool:
    return os.getenv(name, default).strip().lower() in {"1", "true", "yes", "on"}

***TODO:*** The following variables needs to be defined in order to run the code in this notebook:

In [ ]:
# ---- Loading secrets from .env file ----
env_path = Path("./.env").resolve()
load_dotenv(env_path)

# must be https://<host>:<port>/api/catalog
catalog_uri = os.getenv("ICEBERG_CATALOG_URI")
catalog_unsecure = env_bool("ICEBERG_UNSECURE_CONNECTION", "false")

s3_endpoint = os.getenv("S3_ENDPOINT")
s3_region = os.getenv("S3_REGION")

# ---- GUEST CREDENTIALS CONFIGURATION ----
catalog_warehouse = "xxxx"                            # <-- TODO: replace with your warehouse name 
iceberg_namespace = "xxxx"                            # <-- TODO: replace with your namespace in the catalog
# must be in the format "username:password"
catalog_scope = "PRINCIPAL_ROLE:ALL"
catalog_credential = os.getenv("CATALOG_CREDENTIAL")
if not catalog_credential:
    raise ValueError("CATALOG_CREDENTIAL environment variable is not set")


In [4]:
if catalog_unsecure:
    print("Using unsecure connection to catalog, skipping TLS verification")
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

Using unsecure connection to catalog, skipping TLS verification


The following cell is a workaround because the S3 endpoint, despite having a valid certificate, is not recognized as trusted by defautl by the Python `requests` library, which is used by the Iceberg client to connect to the catalog. 
If the connection to the S3 storage is secure, this cell can be removed and the `PyArrowFileIO` can be used instead of the `CephRGWFileIO` class defined in the cell below.

In [ ]:
path_to_cert = "./certs/buckets_areasciencepark_it-chain.pem"

class CephRGWFileIO(PyArrowFileIO):
    def _initialize_s3_fs(self, netloc=None):
        return S3FileSystem(
            anonymous=True,
            endpoint_override=f"{s3_endpoint}:443",
            region=self.properties.get("s3.region"),
            scheme="https",
            force_virtual_addressing=False,
            tls_ca_file_path=f"{path_to_cert}",
        )
        
# # -- use this class in case the S3 buckets need authentication
# class CephRGWFileIO(PyArrowFileIO):
#     def _initialize_s3_fs(self, netloc=None):
#         return S3FileSystem(
#             anonymous=True,
#             endpoint_override=f"{s3_endpoint}:443",
#             region=self.properties.get("s3.region"),
#             access_key=self.properties.get("s3.access-key-id"),
#             secret_key=self.properties.get("s3.secret-access-key"),
#             scheme="https",
#             force_virtual_addressing=False,
#             tls_ca_file_path=f"{path_to_cert}",
#         )

In [ ]:
catalog = load_catalog(
    "polaris",
    type="rest",
    uri=catalog_uri,
    warehouse=catalog_warehouse,
    credential=catalog_credential,
    scope="PRINCIPAL_ROLE:ALL",
    **{
        
        "py-io-impl": "__main__.CephRGWFileIO",
        "ssl": {"cabundle": False} if catalog_unsecure else {},
        "header.Polaris-Realm": "POLARIS",
        "header.X-Iceberg-Access-Delegation": "",
        "s3.endpoint": f"{s3_endpoint}:443",
        "s3.region": s3_region,
        "s3.force-virtual-addressing": "false",
        # # -- Uncomment this and set the right values if the S3 storage requires authentication
        # "s3.access-key-id": <your_s3_access_key>,
        # "s3.secret-access-key": <your_s3_secret_key>,
    },
)

## Perform a query

In [7]:
# ----Try to perform a simple scan on data in the catalog ----
TABLE = "ipmi_sensor"
iceberg_table_name = f"{iceberg_namespace}.{TABLE}"

try:
    table = catalog.load_table(iceberg_table_name)
    print(f"Loaded table: {iceberg_table_name}")
    
    # query: get all records with timestamp between 10:00 and 10:30 on March 5th, 2026
    scanned_table = table.scan(
        row_filter=And(
            GreaterThanOrEqual("timestamp", datetime(2026, 3, 5, 10, 0, 0)),
            LessThan("timestamp", datetime(2026, 3, 5, 10, 30, 0)),
        )
    )
    print("Scan created successfully")

except Exception as e:
    print("Failed while loading or scanning the table")
    print(type(e).__name__, str(e))
    scanned_table = None

Loaded table: raw.ipmi_sensor
Scan created successfully


Example of how to retrieve data in a pandas or polars DataFrame:

In [8]:
# # ---- convert to pandas ----
# df : pd.DataFrame = scanned_table.to_pandas()
# print(df.head())

In [9]:
# ---- convert to polars ----
df_polars : pl.DataFrame = scanned_table.to_polars()
print(df_polars.head())

shape: (5, 5)
┌─────────────────────────────┬────────┬─────────────────────────┬──────┬────────┐
│ timestamp                   ┆ host   ┆ name                    ┆ unit ┆ value  │
│ ---                         ┆ ---    ┆ ---                     ┆ ---  ┆ ---    │
│ datetime[μs, UTC]           ┆ str    ┆ str                     ┆ str  ┆ f64    │
╞═════════════════════════════╪════════╪═════════════════════════╪══════╪════════╡
│ 2026-03-05 10:00:00.148 UTC ┆ ceph23 ┆ Processor_1_Temp        ┆ C    ┆ 55.0   │
│ 2026-03-05 10:00:00.148 UTC ┆ ceph23 ┆ Processor_2_Temp        ┆ C    ┆ 49.0   │
│ 2026-03-05 10:00:00.148 UTC ┆ ceph23 ┆ System_Board_Inlet_Temp ┆ C    ┆ 20.0   │
│ 2026-03-05 10:00:00.148 UTC ┆ ceph23 ┆ System_Board_Fan1A      ┆ RPM  ┆ 5520.0 │
│ 2026-03-05 10:00:00.148 UTC ┆ ceph23 ┆ System_Board_Fan3A      ┆ RPM  ┆ 5400.0 │
└─────────────────────────────┴────────┴─────────────────────────┴──────┴────────┘
